In [3]:
import socket
import ssl
import sys
from urllib.parse import urlparse

def fetch_webpage(url: str):
    # Step 0: Ensure URL contains a scheme prefix
    url = url.strip()
    if not url.startswith("http://") and not url.startswith("https://"):
        url = "https://" + url

    parsed_url = urlparse(url)
    scheme = parsed_url.scheme.lower()
    
    # Clean host domain name (remove ports if present in netloc for DNS lookup)
    host = parsed_url.hostname
    if not host:
        raise ValueError(f"Could not parse valid hostname from URL: {url}")

    path = parsed_url.path if parsed_url.path else "/"
    if parsed_url.query:
        path += f"?{parsed_url.query}"

    # Determine port and SSL requirement
    if scheme == "https":
        port = parsed_url.port if parsed_url.port else 443
        use_ssl = True
    elif scheme == "http":
        port = parsed_url.port if parsed_url.port else 80
        use_ssl = False
    else:
        raise ValueError(f"Unsupported scheme: {scheme}")

    # Step 1: Create raw TCP Socket
    raw_sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)

    if use_ssl:
        # Create unverified SSL context to bypass local Windows CA lookup errors
        context = ssl._create_unverified_context()
        sock = context.wrap_socket(raw_sock, server_hostname=host)
        print("[*] TLS/SSL session initiated (Certificate verification bypassed).")
    else:
        sock = raw_sock

    try:
        print(f"[*] Resolving domain and connecting to {host}:{port} via TCP...")
        sock.connect((host, port))

        # Step 2: Construct HTTP/1.1 Request
        http_request = (
            f"GET {path} HTTP/1.1\r\n"
            f"Host: {host}\r\n"
            f"User-Agent: MinimalSocketBrowser/1.0\r\n"
            f"Accept: text/html,application/xhtml+xml\r\n"
            f"Connection: close\r\n\r\n"
        )

        print("[*] Sending HTTP Request...")
        sock.sendall(http_request.encode('utf-8'))

        # Step 3: Receive Response Stream
        response_data = bytearray()
        while True:
            chunk = sock.recv(4096)
            if not chunk:
                break
            response_data.extend(chunk)

    except socket.gaierror:
        print(f"\n[ERROR] Could not resolve host address for '{host}'. Check your internet connection or domain spelling.")
        return
    finally:
        sock.close()

    # Step 4: Split headers from response body using double CRLF (\r\n\r\n)
    header_data, _, body_data = response_data.partition(b"\r\n\r\n")

    headers_str = header_data.decode('utf-8', errors='replace')
    body_str = body_data.decode('utf-8', errors='replace')

    print("\n[*] HTTP Response Headers:")
    print("--------------------------------------------------")
    print(headers_str)
    print("--------------------------------------------------\n")

    print(f"[*] Response Body ({len(body_data)} bytes received):")
    print("--------------------------------------------------")
    print(body_str[:1000] + ("\n... [truncated]" if len(body_str) > 1000 else ""))
    print("--------------------------------------------------")


if __name__ == "__main__":
    target_url = sys.argv[1] if len(sys.argv) > 1 else "https://example.com"
    fetch_webpage(target_url)

[*] TLS/SSL session initiated (Certificate verification bypassed).
[*] Resolving domain and connecting to -f:443 via TCP...

[ERROR] Could not resolve host address for '-f'. Check your internet connection or domain spelling.
